In [ ]:
# import patch_numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud # type: ignore
import spacy # type: ignore

In [ ]:
# %pip install spacy

In [ ]:
# !python -m spacy download en_core_web_sm

In [ ]:
url = "https://raw.githubusercontent.com/Jcharis/Bible-NLP-and-ML-using-Python/master/kjv_cleandata.csv"
data = pd.read_csv(url)
data.head()

In [ ]:
# extracting named entities
nlp = spacy.load("en_core_web_sm")

def extract_entities(text:str,entity_type: str = 'PERSON'):
  doc = nlp(text)
  entities = [ent.text for ent in doc.ents if ent.label_ == entity_type]
  return entities

In [ ]:
extract_entities("Jesse works with David in Accra","PERSON")

In [ ]:
data["ner_person"] = data["text"].apply(extract_entities)
data.head()

In [ ]:
person_list = [i for item in data['ner_person'].tolist() for i in item]
person_list

In [ ]:
def plot_wordcloud(docx):
  my_wordcloud = WordCloud().generate(docx)
  plt.imshow(my_wordcloud,interpolation="bilinear")
  plt.axis("off")
  plt.show()

In [ ]:
plot_wordcloud(" ".join(person_list))

In [ ]:
from collections import Counter
most_common_name = Counter(person_list)
most_common_name.most_common(20)

In [ ]:
bible_names_data = pd.DataFrame(most_common_name.most_common(20),columns=["names","count"])
bible_names_data.plot(kind="bar")

In [ ]:
# import plotly
# print("Plotly version: ", plotly.__version__)

In [ ]:
# %pip install --upgrade plotly

In [ ]:
# print("Plotly version: ", plotly.__version__)

In [ ]:
import plotly.express as px
px.bar(bible_names_data,x="names",y="count")

In [ ]:
# Network analysis
import networkx as nx

In [ ]:
def get_relationships(clean_data,window_size:int = 5, entity_column:str = 'person'):
  relationships = []
  
  for i in range(clean_data.index[-1]):
    end_i = min(i+5,clean_data.index[-1])
    char_list = sum((clean_data.loc[i:end_i][entity_column]),[])
    
    # Remove duplicated characters that are next to each other
    char_unique = [char_list[i] for i in range(len(char_list)) if (i==0) or char_list[i] != char_list[i-1]]
    
    if len(char_unique) > 1:
      for idx, a in enumerate(char_unique[:-1]):
        b = char_unique[idx + 1]
        relationships.append({"source":a, "target":b})
        
  return relationships 

In [ ]:
data["person"] = data['ner_person'].apply(lambda x: [item.split()[0] for item in x])

In [ ]:
clean_data = data

In [ ]:
relationship_data = get_relationships(clean_data)
relationship_data

In [ ]:
relationship_data = pd.DataFrame(relationship_data)
relationship_data.head()

In [ ]:
relationship_data = pd.DataFrame(np.sort(relationship_data.values,axis=1),columns=relationship_data.columns)
relationship_data

In [ ]:
relationship_data["value"] = 1
relationship_data = relationship_data.groupby(["source","target"],sort=False,as_index=False).sum()
relationship_data